In [29]:
from datetime import date, timedelta
import pandas as pd
import numpy as np
import xgboost as xgb
import json
import os

### Predecir próximos días


In [ ]:
tipos_polinicos = {
    "Gram": "gramineas",
    "Cupres": "cupresaceas",
    "Olivo": "olivo",
    "Plá": "platano",
    "Urtic": "urticaceas",
    "Queno": "quenopodiaceas"
}
for iniciales, tipo in tipos_polinicos.items():

    df = pd.read_csv(rf"..\my_datasets\datos_{tipo}.csv")
    df = df.set_index("fecha")
    df.index = pd.to_datetime(df.index)
    df = df.sort_index(ascending=True)

    with open(f'json/{tipo}/features_t0.json', 'r') as f:
        TOP_FEATURES_T0 = json.load(f)
    model_t0 = xgb.XGBRegressor()
    model_t0.load_model(f"json/{tipo}/modelo_t0.json")

    with open(f'json/{tipo}/features_t1.json', 'r') as f:
        TOP_FEATURES_T1 = json.load(f)
    model_t1 = xgb.XGBRegressor()
    model_t1.load_model(f"json/{tipo}/modelo_t1.json")

    with open(f'json/{tipo}/features_t2.json', 'r') as f:
        TOP_FEATURES_T2 = json.load(f)
    model_t2 = xgb.XGBRegressor()
    model_t2.load_model(f"json/{tipo}/modelo_t2.json")

    # Predictions 3 days
    today_dt = pd.to_datetime(date.today().strftime("%Y-%m-%d"))
    tomorrow_dt = today_dt + pd.Timedelta(days=1)
    after_tomorrow_dt = today_dt + pd.Timedelta(days=2)

    last_10_indices = df.tail(10).index
    for i in last_10_indices:
        if df.loc[i, "isPrediction"] == 1:
            if i <= today_dt:
                features_row = df.loc[[i], TOP_FEATURES_T0]
                prediccion_log = model_t0.predict(features_row)[0]
                prediccion_final = np.expm1(prediccion_log)
                df.loc[i, f"polen_{tipo}"] = round(
                    max(0, prediccion_final), 1
                )
            if i == tomorrow_dt:
                features_row = df.loc[[i], TOP_FEATURES_T1]
                prediccion_log = model_t1.predict(features_row)[0]
                prediccion_final = np.expm1(prediccion_log)
                df.loc[i, f"polen_{tipo}"] = round(
                    max(0, prediccion_final), 1
                )
            if i == after_tomorrow_dt:
                features_row = df.loc[[i], TOP_FEATURES_T2]
                prediccion_log = model_t2.predict(features_row)[0]
                prediccion_final = np.expm1(prediccion_log)
                df.loc[i, f"polen_{tipo}"] = round(
                    max(0, prediccion_final), 1
                )

    print(df[[f"polen_{tipo}", 'isPrediction']].tail(10))
    df.to_csv(rf"..\my_datasets\datos_{tipo}.csv", index=True)

            polen_gramineas  isPrediction
fecha                                    
2026-05-02        65.000000             0
2026-05-03       282.000000             0
2026-05-04       232.000000             0
2026-05-05        19.000000             0
2026-05-06        16.000000             0
2026-05-07        40.000000             0
2026-05-08        15.300000             1
2026-05-09        14.500000             1
2026-05-10        19.700001             1
2026-05-11        32.799999             1
            polen_cupresaceas  isPrediction
fecha                                      
2026-05-02                1.0             0
2026-05-03                0.0             0
2026-05-04                5.0             0
2026-05-05                2.0             0
2026-05-06                3.0             0
2026-05-07                1.0             0
2026-05-08               21.5             1
2026-05-09                9.3             1
2026-05-10                4.3             1
2026-05-11  

### Save Real_data vs Predictions

In [31]:
for iniciales, tipo in tipos_polinicos.items():
    path_comparation = rf"..\my_datasets\comparations\{tipo}.csv"
    path_datos = rf"..\my_datasets\datos_{tipo}.csv"

    if os.path.exists(path_comparation):
        df_comparation = pd.read_csv(path_comparation, index_col=0)
    else:
        df_comparation = pd.DataFrame(columns=["real", "prediccion"])
    
    df = pd.read_csv(path_datos, index_col=0)
    df_last10 = df.tail(10)

    for idx, row in df_last10.iterrows():
        valor = row[f"polen_{tipo}"]
        if row["isPrediction"] == 1:
            df_comparation.loc[idx, "prediccion"] = valor
        else:
            df_comparation.loc[idx, "real"] = valor
    
    df_comparation.to_csv(rf"..\my_datasets\comparations\{tipo}.csv", index=True)

### Pollutants JSON

In [32]:
CSV_POLEN = r"..\my_datasets\datos_gramineas.csv"
df = pd.read_csv(CSV_POLEN)
today = date.today().strftime('%Y-%m-%d')

df_today = df[df['fecha'] == today]

clean_names = {
    "NO2 (ug/m3)": "NO2",
    "O3 (ug/m3)": "O3",
    "PM10 (ug/m3)": "PM10",
    "PM2.5 (ug/m3)": "PM2.5",
    "CO (mg/m3)": "CO",
    "SO2 (ug/m3)": "SO2"
}
datos = df_today[list(clean_names.keys())].iloc[0].to_dict()
pollutants_json = {clean_names[k]: round(float(v), 1) for k, v in datos.items()}

folder_path = r"..\..\Prediction-service"
file_name = "pollutants.json"
full_path = os.path.join(folder_path, file_name)

os.makedirs(folder_path, exist_ok=True)
with open(full_path, 'w', encoding='utf-8') as f:
    json.dump(pollutants_json, f, indent=4, ensure_ascii=False)


### Polen Prediction JSON

In [33]:
tipos_polinicos = {
    "Gramineas": "gramineas",
    "Cupresacea": "cupresaceas",
    "Olivo": "olivo",
    "Platano_de_paseo": "platano",
    "Urticaceas": "urticaceas",
    "Quenopodiaceas": "quenopodiaceas"
}

def get_val(df, fecha):
    try:
        valor = df.loc[fecha, f'polen_{tipo}']
        if pd.isna(valor):
            return 0.0
        else:
            return float(valor.iloc[0]) if hasattr(valor, 'iloc') else float(valor)
    except Exception:
        return 0.0

def get_full_obj(target_dt):
    val = get_val(df, target_dt)
    return {
        "value": round(val, 1) if val is not None else 0,
        "isPrediction": bool(df.loc[target_dt, 'isPrediction'] if target_dt in df.index else True)
    }

folder_path = r"..\..\Prediction-service"
file_name = "polen.json"
full_path = os.path.join(folder_path, file_name)
with open(full_path, 'r', encoding='utf-8') as f:
    json_polen = json.load(f)

for jsonName, tipo in tipos_polinicos.items():
    CSV_POLEN = rf"..\my_datasets\datos_{tipo}.csv"
    df = pd.read_csv(CSV_POLEN)
    today = date.today().strftime('%Y-%m-%d')

    today_dt = pd.to_datetime(date.today())
    tomorrow_dt = today_dt + timedelta(days=1)
    after_tomorrow_dt = today_dt + timedelta(days=2)

    df['fecha'] = pd.to_datetime(df['fecha'])
    today_dt = pd.to_datetime(date.today())
    df_lastWeek = df[df['fecha'] < today_dt].tail(7)

    df = df.set_index('fecha')
    data = {
        "historical": [get_full_obj(d) for d in df_lastWeek['fecha']],
        "prediction": {
            "today": get_full_obj(today_dt),
            "tomorrow" : get_full_obj(tomorrow_dt),
            "day_after_tomorrow": get_full_obj(after_tomorrow_dt)
            }
        }
    json_polen[jsonName] = data

with open(full_path, 'w', encoding='utf-8') as f:
    json.dump(json_polen, f, indent=4, ensure_ascii=False)